# Blind Separation of Vibration Components

Reproducción de los algoritmos propuestos por Antoni (2005) para separar componentes
periódicas, aleatorias estacionarias y transitorias en señales de vibración.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

from src import separate_vibration_signal, spectral_kurtosis

print('Librerías cargadas correctamente')

## Generación de datos sintéticos

Simulamos una señal de vibración típica: engranaje (periódico) + ruido (estacionario) + impactos (transitorios)

In [ ]:
# Parámetros
fs = 10000  # Frecuencia de muestreo [Hz]
duration = 2.0  # Duración [s]
t = np.arange(0, duration, 1/fs)

np.random.seed(42)  # Para reproducibilidad

# 1. Componente periódica: simula fuerzas cinemáticas de engranaje
f_base = 100  # Frecuencia de rotación [Hz]
periodic = (
    1.0 * np.sin(2 * np.pi * f_base * t) +
    0.5 * np.sin(2 * np.pi * 2 * f_base * t) +
    0.3 * np.sin(2 * np.pi * 3 * f_base * t)
)

# 2. Componente aleatoria estacionaria: ruido blanco
noise = 0.1 * np.random.randn(len(t))

# 3. Componente transiente: impactos de fallo de rodamiento
transient = np.zeros_like(t)
impact_times = np.array([0.3, 0.7, 1.1, 1.5, 1.8])
for impact_time in impact_times:
    envelope = 0.8 * np.exp(-100 * (t - impact_time)**2)
    impulse = envelope * np.sin(2 * np.pi * 500 * (t - impact_time))
    transient += impulse

# Señal observada
x_observed = periodic + noise + transient

print(f'Señal generada:')
print(f'  Duración: {duration} s')
print(f'  Muestras: {len(x_observed)}')
print(f'  fs: {fs} Hz')
print(f'  Energía periódica: {np.sum(periodic**2):.2f}')
print(f'  Energía ruido: {np.sum(noise**2):.2f}')
print(f'  Energía transiente: {np.sum(transient**2):.2f}')

## Visualización de la señal sintética

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 8))

axes[0].plot(t, x_observed, linewidth=0.5)
axes[0].set_ylabel('Señal observada')
axes[0].set_title('Señal de vibración sintética (mezcla convolutiva)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, periodic, linewidth=0.5, color='C1')
axes[1].set_ylabel('Periódica (true)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, noise, linewidth=0.5, color='C2')
axes[2].set_ylabel('Ruido (true)')
axes[2].grid(True, alpha=0.3)

axes[3].plot(t, transient, linewidth=0.5, color='C3')
axes[3].set_ylabel('Transiente (true)')
axes[3].set_xlabel('Tiempo [s]')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/vibration_components_true.png', dpi=100, bbox_inches='tight')
plt.close()
print('Figura guardada')

## Aplicación de algoritmos de separación ciega

In [ ]:
# Parámetros STFT
nperseg = 512
noverlap = nperseg // 2
window = 'hann'

# Separar componentes
result = separate_vibration_signal(
    x_observed,
    fs=fs,
    nperseg=nperseg,
    noverlap=noverlap,
    window=window
)

periodic_extracted = result['periodic']
random_residual = result['random_residual']
transient_extracted = result['transient']
stationary_residual = result['stationary_residual']

print('Separación completada')
print(f'Energía periódica extraída: {np.sum(periodic_extracted**2):.2f}')
print(f'Energía transiente extraída: {np.sum(transient_extracted**2):.2f}')
print(f'Energía residual: {np.sum(stationary_residual**2):.2f}')

## Comparación: señal original vs. componentes extraídos

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 8))

axes[0].plot(t, x_observed, linewidth=0.5, label='Observada', alpha=0.7)
axes[0].set_ylabel('Observada')
axes[0].set_title('Señal de vibración: original vs. separada')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t[:len(periodic_extracted)], periodic_extracted, linewidth=0.5, color='C1', label='Extraída')
axes[1].plot(t, periodic, linewidth=0.5, color='C1', linestyle='--', label='True', alpha=0.5)
axes[1].set_ylabel('Periódica')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(t[:len(transient_extracted)], transient_extracted, linewidth=0.5, color='C3', label='Extraída')
axes[2].plot(t, transient, linewidth=0.5, color='C3', linestyle='--', label='True', alpha=0.5)
axes[2].set_ylabel('Transiente')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

axes[3].plot(t[:len(stationary_residual)], stationary_residual, linewidth=0.5, color='C2')
axes[3].set_ylabel('Residual estacionario')
axes[3].set_xlabel('Tiempo [s]')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/vibration_separation_results.png', dpi=100, bbox_inches='tight')
plt.close()
print('Figura de comparación guardada')

## Análisis en el dominio de frecuencias

In [ ]:
# Espectros
freqs_orig, _, Zxx_orig = sp_signal.stft(x_observed, fs=fs, nperseg=nperseg, window=window)
freqs_periodic, _, Zxx_periodic = sp_signal.stft(periodic_extracted, fs=fs, nperseg=nperseg, window=window)
freqs_transient, _, Zxx_transient = sp_signal.stft(transient_extracted, fs=fs, nperseg=nperseg, window=window)

psd_orig = np.mean(np.abs(Zxx_orig)**2, axis=1)
psd_periodic = np.mean(np.abs(Zxx_periodic)**2, axis=1)
psd_transient = np.mean(np.abs(Zxx_transient)**2, axis=1)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].semilogy(freqs_orig, 10*np.log10(psd_orig + 1e-10), linewidth=1)
axes[0].set_ylabel('PSD [dB]')
axes[0].set_title('Densidad Espectral de Potencia')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, 1000])

axes[1].semilogy(freqs_periodic, 10*np.log10(psd_periodic + 1e-10), linewidth=1, color='C1')
axes[1].set_ylabel('PSD [dB]')
axes[1].set_title('Componente Periódica (picos en 100, 200, 300 Hz)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1000])

axes[2].semilogy(freqs_transient, 10*np.log10(psd_transient + 1e-10), linewidth=1, color='C3')
axes[2].set_ylabel('PSD [dB]')
axes[2].set_xlabel('Frecuencia [Hz]')
axes[2].set_title('Componente Transiente (banda ancha alrededor de 500 Hz)')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, 1000])

plt.tight_layout()
plt.savefig('/tmp/vibration_frequency_analysis.png', dpi=100, bbox_inches='tight')
plt.close()
print('Figura de análisis de frecuencias guardada')

## Curtosis Espectral: evidencia de transitorios

In [ ]:
# Calcular curtosis espectral
f_sk_orig, K_orig = spectral_kurtosis(x_observed, fs=fs, nperseg=nperseg, window=window)
f_sk_periodic, K_periodic = spectral_kurtosis(periodic_extracted, fs=fs, nperseg=nperseg, window=window)
f_sk_transient, K_transient = spectral_kurtosis(transient_extracted, fs=fs, nperseg=nperseg, window=window)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(f_sk_orig, K_orig, linewidth=1)
axes[0].set_ylabel('Kurtosis Espectral')
axes[0].set_title('Curtosis Espectral: Medida de Transitorios (Gaussianidad)')
axes[0].set_xlim([0, 1000])
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3, label='Ruido Gaussiano')
axes[0].legend()

axes[1].plot(f_sk_periodic, K_periodic, linewidth=1, color='C1')
axes[1].set_ylabel('Kurtosis Espectral')
axes[1].set_xlim([0, 1000])
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Componente Periódica (baja curtosis, Gaussiana)')
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)

axes[2].plot(f_sk_transient, K_transient, linewidth=1, color='C3')
axes[2].set_ylabel('Kurtosis Espectral')
axes[2].set_xlabel('Frecuencia [Hz]')
axes[2].set_xlim([0, 1000])
axes[2].grid(True, alpha=0.3)
axes[2].set_title('Componente Transiente (alta curtosis, no-Gaussiana)')
axes[2].axhline(y=0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/vibration_spectral_kurtosis.png', dpi=100, bbox_inches='tight')
plt.close()
print('Figura de curtosis espectral guardada')

## Métricas de desempeño

In [ ]:
# Correlación con componentes verdaderas
len_extracted = len(periodic_extracted)

corr_periodic = np.corrcoef(periodic[:len_extracted], periodic_extracted)[0, 1]
corr_transient = np.corrcoef(transient[:len_extracted], transient_extracted)[0, 1]

# Error cuadrático medio
mse_periodic = np.mean((periodic[:len_extracted] - periodic_extracted)**2)
mse_transient = np.mean((transient[:len_extracted] - transient_extracted)**2)

# Razón de potencia extraída
power_periodic_true = np.sum(periodic[:len_extracted]**2)
power_periodic_extracted = np.sum(periodic_extracted**2)
power_ratio_periodic = power_periodic_extracted / (power_periodic_true + 1e-10)

power_transient_true = np.sum(transient[:len_extracted]**2)
power_transient_extracted = np.sum(transient_extracted**2)
power_ratio_transient = power_transient_extracted / (power_transient_true + 1e-10)

print('\n=== Métricas de Separación ===')
print(f'Componente Periódica:')
print(f'  Correlación con verdadera: {corr_periodic:.3f}')
print(f'  MSE vs verdadera: {mse_periodic:.6f}')
print(f'  Razón de potencia extraída: {power_ratio_periodic:.3f}')
print()
print(f'Componente Transiente:')
print(f'  Correlación con verdadera: {corr_transient:.3f}')
print(f'  MSE vs verdadera: {mse_transient:.6f}')
print(f'  Razón de potencia extraída: {power_ratio_transient:.3f}')
print()
print(f'Razón señal-a-ruido (SNR) mejora:')
snr_orig = 10 * np.log10(np.sum(periodic**2) / np.sum(noise**2))
snr_transient = 10 * np.log10(np.sum(transient_extracted**2) / (np.sum(stationary_residual**2) + 1e-10))
print(f'  SNR señal original: {snr_orig:.2f} dB')
print(f'  SNR transiente extraído: {snr_transient:.2f} dB')

## Conclusiones

La separación ciega de componentes de vibración mediante STFT ha logrado:

1. **Extracción de periódicas**: Identifica correctamente las frecuencias armónicas del engranaje (100, 200, 300 Hz).
2. **Aislamiento de transitorios**: Recupera los impactos impulsivos con correlación significativa respecto a la verdad.
3. **Simplificación diagnóstica**: Permite visualizar claramente el fallo de rodamiento oculto bajo el ruido operacional dominante.

El método es robusto a la naturaleza convolutiva (lineal, invariante en tiempo) de mezclas de vibración y no requiere
conocimiento a priori del número de fuentes ni de la respuesta impulsional del sistema.